In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import jax

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")

import jax.numpy as jnp
import numpy as np

from ajx.constraints import ConstraintType
from ajx.example_environments.dlo_attached import DLOAttached, DLOAttachedSettings
from ajx.example_graphics.application import Application
from ajx.example_graphics.environment_scene import EnvironmentScene
from ajx.simulation import SimulationSettings, Solver

N_BODIES = 25
BODY_LENGTH = 0.03

def setup_dlo_environment(timestep, pgs_iterations, solver, youngs_modulus, poission_ratio, mass_density):

    env = DLOAttached(
        sim_settings=SimulationSettings(
            timestep, True, solver, pgs_iterations
        ),
        env_settings=DLOAttachedSettings(
            n_bodies=N_BODIES,
            body_length=BODY_LENGTH,
            mass_density=mass_density,
            constraint_type=ConstraintType.BEND_TWIST.value,
            hinge_motor_attachment=True,
            body_side_length=2*BODY_LENGTH
        ),
    )
    
    shear_modulus = youngs_modulus / (2 * (1 + poission_ratio))
    linear_stiffness, bend_stiffness, torsion_stiffness = env.get_stiffness_from_material_parameters(youngs_modulus, shear_modulus)

    yz_linear_stiffness = linear_stiffness
    x_linear_stiffness = linear_stiffness
    bend_linear_stiffness = bend_stiffness
    torsion_linear_stiffness = torsion_stiffness

    env_param = env.default_param.tree_replace(
        src={
            "sparse_param.coupled_constraint_param": {
                "linear_stiffness.data": jnp.array(
                    [
                        x_linear_stiffness,
                        yz_linear_stiffness,
                        yz_linear_stiffness,
                        bend_linear_stiffness,
                        bend_linear_stiffness,
                        torsion_linear_stiffness,
                    ]
                ),
                "is_velocity": jnp.array([0, 0, 0, 0, 0, 0], dtype=bool),
            }
        }
    )

    return env, env_param


def simulate_dlo(settings_dict, control_signal, tmax=None):
    """
    INPUTS:
        settings_dict: A python dict with settings.
        control_signal: Array of size horizon x 1
        tmax (Optional): Stop simulation if t > tmax
    OUTPUTS:
        dlo_x: x-position data for DLO. array of size (horizon+1) x N_BODIES
        dlo_z: z_position data for DLO. array of size (horizon+1) x N_BODIES
        dz: z_position data for end point of DLO. array of size (horizon + 1) x 1
    """

    # To extract simulation settings/arguments
    horizon = settings_dict["horizon"]
    timestep = settings_dict["timestep"]
    pgs_iterations = settings_dict["pgs_iterations"]
    solver = settings_dict["solver"]
    youngs_modulus = settings_dict["youngs_modulus"]
    poisson_ratio = settings_dict["poisson_ratio"]
    mass_density = settings_dict["mass_density"]

    # To setup and initialize a dlo environment
    env, env_param = setup_dlo_environment(timestep, pgs_iterations, solver, youngs_modulus, poisson_ratio, mass_density)

    state = env.state_from_angles(env_param)
    env_step = jax.jit(env.step)
    u = np.array(control_signal + 1e-12).reshape(-1,1)   # Workaround, add a small number to trigger this being a velocity constraint.

    # To prepare storage of rigid body data
    dlo_pos_x = [state.conf.pos[:,0]]
    dlo_pos_z = [state.conf.pos[:,2]]
    loose_end_id = env_param.rigid_body_param.names.index(f"body{N_BODIES-1}")

    # Simulation loop
    for j in range(horizon):

        if tmax is not None and (j+1)*timestep > tmax:
            break

        # Step the environment and store the observation
        state, observations = env_step(state, u[j], env_param)
        dlo_pos_x.append(state.conf.pos[:,0])
        dlo_pos_z.append(state.conf.pos[:,2])

    dlo_x = np.array(dlo_pos_x)
    dlo_z = np.array(dlo_pos_z)
    dz = dlo_z[:,loose_end_id]
    return dlo_x, dlo_z, dz

## Hinge motor with sinusoidal control signal

In [5]:
settings_dict = {
    "horizon": 500,
    "timestep": 0.01,
    "pgs_iterations": 3000,
    "solver": Solver.DENSE_PGS,
    "youngs_modulus": 1e7,
    "poisson_ratio": 0.3,
    "mass_density": 1000.0,
}

mu_r = 0.1
os.environ["MU_TIMES_EFFECTIVE_RADIUS"] = str(mu_r)     # This is passed to the solver in projected_gauss_seidel.py

period = 3
control_signal = np.zeros([settings_dict["horizon"]])  # (np.pi/period)*np.pi*np.sin(2*np.pi/period * np.arange(settings_dict["horizon"])*settings_dict["timestep"])

dlo_x, dlo_z, _ = simulate_dlo(settings_dict, control_signal)

# To store results for plotting 
dlo_position = {"x": np.array(dlo_x), "z": np.array(dlo_z)}

jax.clear_caches()  # Workaround

In [6]:
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib as mpl
from IPython.display import HTML

def visualize_dlo(dlo_x, dlo_z):
    
    # To precompute axis limits
    x_direct = dlo_x
    z_direct = dlo_z
    x_min = np.min([x_direct.min(), ])
    x_max = np.max([x_direct.max(), ])
    z_min = np.min([z_direct.min(), ])
    z_max = np.max([z_direct.max(), ])

    # Make a figure and use it for an animation
    fig, ax = plt.subplots()
    ax.set_xlabel("x")
    ax.set_ylabel("z")
    ax.set_xlim([x_min-0.05, x_max + 0.1])
    ax.set_ylim([z_min - 0.1, z_max + 0.1])
    ax.grid(True)
    ax.set_aspect('equal')

    # Initialize lines for both direct and iterative solver
    dlo_line, = ax.plot([], [], '-', linewidth=4)

    def update(t):
        dlo_line.set_data(
            dlo_x[t,:],
            dlo_z[t, :]
        )

        return [dlo_line,]

    # Display animation
    mpl.rcParams['animation.embed_limit'] = 50
    anim = FuncAnimation(fig, update, frames=settings_dict["horizon"], interval=settings_dict["timestep"]*1000, blit=True)
    plt.close(fig)

    return anim

anim = visualize_dlo(dlo_position["x"], dlo_position["z"])
HTML(anim.to_jshtml())